In [8]:
from enum import Enum
from dataclasses import dataclass
from datetime import datetime
from matplotlib import pyplot as plt
import pandas as pd
import re

## Задание 1.
Реализуйте базовый класс Account, который моделирует поведение
банковского счёта. Этот класс должен не только выполнять базовые
операции, но и вести детальный учёт всех действий, а также предоставлять аналитику по истории операций.

### Этап 1. Реализация базового класса Account
Класс должен быть инициализирован с параметрами:
- account_holder (str) — имя владельца счёта;
- balance (float, по умолчанию 0) — начальный баланс счёта, не может
быть отрицательным.
Атрибуты:
- _account_counter — приватный атрибут для хранения количества
созданных счетов. Отсчет начинается с 1000;
- holder — хранит имя владельца;
- account_number — хранит номер счёта;
- _balance — приватный атрибут для хранения текущего баланса;
- operations_history — список или другая структура для хранения
истории операций.

*Важно*: каждая операция должна храниться не просто как число, а как
структурированная информация, например, словарь, кортеж или класс.
Минимальный набор данных для операции: тип операции ('deposit' или
'withdraw'), сумма, дата и время операции, текущий баланс после операции, статус ('success' или 'fail').


### Этап 2. Реализация методов
1. __init__(self, account_holder, balance=0) — конструктор. Обратите
внимание, что в конструкторе должен автоматически формироваться
номер счёта в формате ‘ACC-XXXX’, где XXXX — порядковый номер
счёта;
2. deposit(self, amount) — метод для пополнения счёта:
- принимает сумму (должна быть положительной), попытка
положить отрицательную сумму, должна вызывать исключение;
- в случае успеха обновляет баланс и добавляет запись в историю
операций.
3. withdraw(self, amount) — метод для снятия средств:
- принимает сумму (должна быть положительной);
- проверяет, достаточно ли средств на счёте, если нет — операция
не проходит, но ее попытка с статусом 'fail' все равно фиксируется
в истории;
- в случае успеха обновляет баланс и добавляет запись в историю.
4. get_balance(self) — метод, который возвращает текущий баланс.
5. get_history(self) — метод, который возвращает историю операций

*Важно*: продумайте, в каком формате его вернуть. Для работы с датой и
временем используйте модуль datetime. Получить текущее время можно с
помощью datetime.now()

### Этап 3. Визуализация истории операций. Дополнительное задание для
претендующих на оценку 8 и выше баллов (выполняется по желанию).
1. Создайте метод plot_history(self), который использует библиотеку
Pandas для создания датафрейма из истории операций.
2. Продумайте, с помощью какой библиотеки можно отобразить
изменение баланса с течением времени. Постройте простой
линейный график, где по оси X будет время операции, а по оси Y —
баланс после каждой операции. График должен иметь заголовок,
подписи осей

In [ ]:
class OperationType(Enum):
    DEPOSIT = "deposit"
    WITHDRAW = "withdraw"


class OperationStatus(Enum):
    SUCCESS = "success"
    FAIL = "fail"


@dataclass
class Operation:
    operation_type: OperationType
    operation_status: OperationStatus
    timestamp: datetime
    amount: float
    balance_after: float


class Account:
    account_number: str
    holder: str
    operations_history: list[Operation]
    _account_counter: int = 1000
    _balance: float

    
    def __init__(self, account_holder: str, balance: float = 0):
        if not self._validate_name(account_holder):
            raise ValueError(
                "The account holder's name must be in the format 'First Last'" \
                " with capitalized letters (Cyrillic or Latin)")

        if balance < 0:
            raise ValueError("The deposit amount must be positive.")
        
        self.holder = account_holder
        self._balance = balance
        self.account_number = f'ACC-{Account._account_counter}'
        Account._account_counter += 1

    
    @staticmethod
    def _validate_name(name: str) -> bool:
        pattern = re.compile(
            r"^([A-ZА-Я][a-zа-яё]+)\s([A-ZА-Я][a-zа-яё]+)$"
        )
        return bool(pattern.match(name))

    
    def deposit(self, amount: float):
        if amount <= 0:
            raise ValueError("The deposit amount must be positive")
        
        self._balance += amount
        
        operation = Operation(
            operation_type=OperationType.DEPOSIT,
            operation_status=OperationStatus.SUCCESS,
            timestamp=datetime.now(),
            amount=amount,
            balance_after=self._balance
        )
        self.operations_history.append(operation)

    
    def withdraw(self, amount: float):
        if amount <= 0:
            raise ValueError("The deposit amount must be positive")

        if self._balance < amount:
            operation_status = OperationStatus.FAIL
        else:
            operation_status = OperationStatus.SUCCESS
            self._balance -= amount
        
        operation = Operation(
            operation_type=OperationType.WITHDRAW,
            operation_status=operation_status,
            timestamp=datetime.now(),
            amount=amount,
            balance_after=self._balance
        )
        self.operations_history.append(operation)


    def get_balance(self) -> float:
        return self._balance
    

    def get_history(self) -> pd.DataFrame:
        data = [{
            "type": op.operation_type.value,
            "amount": op.amount,
            "time": op.timestamp,
            "balance_after": op.balance_after,
            "status": op.status.value,
        } for op in self.operations_history]
        
        return pd.DataFrame(data)
    

    def plot_histry(self):
        history = self.get_history()

        plt.figure(figsize=(8, 4))
        plt.plot(history["time"], history["balance_after"], marker="o", linestyle="-")
        plt.title(f"Изменение баланса для счета {self.account_number}")
        plt.xlabel("Время операции")
        plt.ylabel("Баланс после операции")
        plt.grid(True)
        plt.tight_layout()
        plt.show()

In [ ]:
class AccountType(Enum):
    CHECKING_ACCOUNT = "checking_account"
    SAVINGS_ACCOUNT = "savings_account"


class CheckingAccount(Account):
    account_type: AccountType = AccountType.CHECKING_ACCOUNT
    

class SavingsAccount(Account):
    account_type: AccountType = AccountType.SAVINGS_ACCOUNT
    

    def apply_intereset(self, rate: float) -> float:
        return self._balance * rate / 100


    def withdraw(self, amount: float):
        if amount <= 0:
            raise ValueError("The deposit amount must be positive")

        if self._balance * 0.5 < amount:
            operation_status = OperationStatus.FAIL
        else:
            operation_status = OperationStatus.SUCCESS
            self._balance -= amount
        
        operation = Operation(
            operation_type=OperationType.WITHDRAW,
            operation_status=operation_status,
            timestamp=datetime.now(),
            amount=amount,
            balance_after=self._balance
        )
        self.operations_history.append(operation)